In [1]:
import pandas as pd
import numpy as np
import robustsharpe as rs

In [4]:
average_model_path = "../result/grpo/03_total_avg.csv"
equal_strategy_path = "../result/benchmark/0.003_benchmark.csv"

In [5]:
grpo_best = pd.read_csv(average_model_path, index_col=0)

In [6]:
grpo_best

,GRPO,PPO,SAC
2019-01-30,0.014351,0.019297,0.051015
2019-02-28,0.017790,0.001676,0.015721
2019-03-28,-0.006875,-0.009401,-0.007927
2019-04-26,0.022650,0.021205,0.017539
2019-05-24,-0.025741,-0.027819,-0.022284
...,...,...,...
2024-09-20,0.025878,0.001493,0.036039
2024-10-18,0.000756,0.016962,0.002895
2024-11-15,-0.023999,-0.020194,-0.033264
2024-12-16,0.062063,0.078830,0.022731


In [9]:
eqaul = pd.read_csv(equal_strategy_path, index_col=0)

In [27]:
my_strategy_returns = np.array(grpo_best["SAC"])

In [28]:
benchmark_returns = np.array(eqaul["equal_asset_weight"])

In [29]:
returns = np.stack([my_strategy_returns, benchmark_returns], axis=1)  # shape: (T, 2)

In [30]:
# returns = np.stack([benchmark_returns, my_strategy_returns], axis=1)  # shape: (T, 2)

In [31]:
# best_block_scores = rs.block_size_calibrate(
#     returns=returns,         # shape (T, 2)
#     b_vec=[1, 2, 4, 6],  # 후보 block size
#     alpha=0.05,
#     M=199,                    # bootstrap per test
#     K=1000,                    # pseudo-sequence 생성 횟수
#     T_start= 20
# )


In [32]:
b_vec=[1, 2, 3, 4, 5, 6]

In [33]:
# best_block_scores = rs.block_size_calibrate(
#     returns=returns,
#     b_vec=[1, 2, 3, 4, 5, 6],  # T=76이므로 6 이상은 피하는게 좋음
#     alpha=0.05,
#     K=300,
#     M=99,
#     T_start=20
# )


In [34]:
SRs, diff, ci, pval, se, d = rs.bootstrap_inference(
    returns=returns,
    block_size=4,     # 논문 추천값 (T=120 기준)
    alpha=0.05,
    M=1000
)

print("Sharpe ratio (벤치마크, 내):", SRs)
print("Sharpe ratio 차이:", diff)
print("95% 신뢰구간:", ci)
print("t-통계량", d)
print("p-value:", pval)

Sharpe ratio (벤치마크, 내): [0.19534707 0.14552373]
Sharpe ratio 차이: -0.04982334274487696
95% 신뢰구간: (np.float64(-0.17488195173249985), np.float64(0.07523526624274593))
t-통계량 0.8598345615777104
p-value: 0.42657342657342656


In [46]:
diff / se

np.float64(-1.5650429769955672)

In [47]:
eqaul["equal_asset_weight"]

2019-01-30    0.054332
2019-02-28    0.016301
2019-03-28   -0.004392
2019-04-26    0.024158
2019-05-24   -0.027220
                ...   
2024-09-20    0.034407
2024-10-18    0.002778
2024-11-15   -0.020486
2024-12-16    0.036143
2024-12-31   -0.022752
Name: equal_asset_weight, Length: 76, dtype: float64